# Experiment 3 — class-imbalance robustness (PD)

How AUC and the prevalence-corrected **AP_normalized** hold up as the minority-class proportion is driven down (no HPO; the four leading methods). Pooled curves in four views for each metric, then per-dataset raw points, a degradation table, and a summary of both **trajectories**. Figures → `figures/experiment3/`.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import results_root
from src.methods.method_config import HPO_METHODS          # tuned methods -> HPO time x n_trials
from src.visualizations.experiment_plots import (
    apply_style, reset_figure_dir, load_summary,
    performance_heatmap, rank_heatmap, hpo_improvement_bars,
    compute_time_bars, runtime_performance_scatter,
    learning_curve, imbalance_curve, per_dataset_sweep_curves,
    sweep_evolution_summary, pd_summary_text, lgd_summary_text,
)
apply_style()

RESULTS_ROOT = results_root()          # override here if your copy lives elsewhere
SUMMARY_DIR  = RESULTS_ROOT / 'summaries'
FIGURES_DIR  = reset_figure_dir(PROJECT_ROOT / 'figures' / 'experiment3')
print('results:', RESULTS_ROOT, '| figures:', FIGURES_DIR)

In [ ]:
df = load_summary(SUMMARY_DIR, experiment='experiment3', task='pd', aggregated=False)
print(f'{df["method"].nunique()} methods, {df["dataset"].nunique()} datasets, '
      f'{df["sweep_value"].nunique()} sweep points')

## 1. AUC vs minority-class proportion — pooled over datasets

Four views (one line per method, mean over datasets): raw points, moving average, relative to each method's own best, and the moving average of that relative curve.

In [ ]:
imbalance_curve(df, 'AUC', task_name='PD', out_dir=FIGURES_DIR)

In [ ]:
imbalance_curve(df, 'AUC', task_name='PD', smooth=True, out_dir=FIGURES_DIR)

In [ ]:
imbalance_curve(df, 'AUC', task_name='PD', relative=True, out_dir=FIGURES_DIR)

In [ ]:
imbalance_curve(df, 'AUC', task_name='PD', relative=True, smooth=True, out_dir=FIGURES_DIR)

## 2. AP_normalized (prevalence-corrected) — pooled over datasets

Same four views. AP_normalized = (AP − π)/(1 − π) removes the prevalence baseline, so it is comparable across imbalance levels and datasets.

In [ ]:
imbalance_curve(df, 'AP_normalized', task_name='PD', out_dir=FIGURES_DIR)

In [ ]:
imbalance_curve(df, 'AP_normalized', task_name='PD', smooth=True, out_dir=FIGURES_DIR)

In [ ]:
imbalance_curve(df, 'AP_normalized', task_name='PD', relative=True, out_dir=FIGURES_DIR)

In [ ]:
imbalance_curve(df, 'AP_normalized', task_name='PD', relative=True, smooth=True, out_dir=FIGURES_DIR)

## 3. AUC per dataset (raw points, all methods)

One plot per dataset.

In [ ]:
per_dataset_sweep_curves(df, 'AUC', sweep_axis='minority_proportion', xlabel='Minority-class proportion', task_name='PD', out_dir=FIGURES_DIR)

## 4. Degradation — AUC at the easiest minus the hardest setting

In [ ]:
import pandas as _pd
g = df.groupby(['method','sweep_value'])['metric.AUC'].mean().reset_index()
drop = {m: gg.sort_values('sweep_value').iloc[-1]['metric.AUC']
            - gg.sort_values('sweep_value').iloc[0]['metric.AUC']
        for m, gg in g.groupby('method')}
display(_pd.Series(drop, name='AUC(max minority) - AUC(min minority)').sort_values(ascending=False))

## 5. Summary — AUC & AP_normalized evolution over minority proportion

Mean per method across all datasets at a spread of minority-class proportions (not just the most balanced point), so the whole robustness trajectory is visible.

In [ ]:
sweep_evolution_summary(df, 'AUC', sweep_axis='minority_proportion', task_name='Experiment 3 — PD')

In [ ]:
sweep_evolution_summary(df, 'AP_normalized', sweep_axis='minority_proportion', task_name='Experiment 3 — PD')